# RTM–WB Introduction: A Local Circuit in Each Area

This notebook pairs one RTM excitatory cell with one Wang–Buzsaki (WB) inhibitory interneuron in each named brain area. The two cells interact through local excitatory and inhibitory synapses. V4 receives the external current pulse. There are still no connections between different brain areas.


## Step 1: Import Tools and Locate the Project

Locate the project root and import Brian2 and the tools for reading area data. If a module cannot be found, make sure Jupyter is using the project's .venv environment.


In [ ]:
from pathlib import Path
import sys
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data").is_dir() and (p / "template_kwargs.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter from the project root or one of its subdirectories.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt
from brian2 import ms
from template_kwargs import template_kwargs
from model_class import RTMAreaModel
print("Project root:", project_root)


## Step 2: Read the Area List

The area_list in the connectivity data is used only to get area names and the number of areas. FLN is not used to create connections between areas here.


In [ ]:
connectivity_path = project_root / "data" / template_kwargs["connectivity_file"]
with open(connectivity_path, "rb") as f:
    anatomy = np.load(f, allow_pickle=True, encoding="bytes")
area_names = anatomy[b"area_list"]
area_labels = [
    x.decode("utf-8") if isinstance(x, bytes) else str(x) for x in area_names
]
n_areas = len(area_labels)
print("Number of areas:", n_areas)
print("First few areas:", area_labels[:10])


## Step 3: Set the Simulation Duration and Input

The stimulus is applied to the RTM cell in V4. WB cells do not receive direct external stimulation; they are driven by local RTM-to-WB synapses. The MATLAB model does not state the current units, so the input is provisionally interpreted as µA/cm².


In [ ]:
run_duration = 500 * ms
integration_dt = 0.01 * ms
base_current = 0.0
pulse_current = 0.24
pulse_on = 100 * ms
pulse_off = 5000 * ms
print("Run duration:", run_duration / ms, "ms")
print("Integration time step:", integration_dt / ms, "ms")
print("Pulse starts at:", pulse_on / ms, "ms")


## Step 4: RTM and WB Cell Dynamics

The RTM equations describe the excitatory cell. The WB equations describe a fast-spiking inhibitory interneuron. Each cell has its own membrane voltage and gating variables. The WB cell receives excitation from the local RTM cell; its spikes then activate an inhibitory conductance on that RTM cell.


In [ ]:
print('RTM equations:\n', RTMAreaModel.equations)
print('\nWB equations:\n', RTMAreaModel.wb_equations)


## Step 5: Create One Local RTM–WB Pair per Area

Each area gets one RTM excitatory cell and one WB inhibitory cell. They connect only to each other: RTM spikes excite WB, and WB spikes inhibit RTM. V4 alone receives the external pulse. This is a minimal local-circuit demonstration, not yet a multi-neuron population model.


In [ ]:
rtm_model = RTMAreaModel(
    area_labels,
    run_duration=run_duration,
    integration_dt=integration_dt,
    base_current=base_current,
    pulse_current=pulse_current,
    pulse_on=pulse_on,
    pulse_off=pulse_off,
    stimulated_areas=['V4'],
)
v4_index = rtm_model.area_names.index('V4')
print('RTM excitatory cells:', len(rtm_model.area_names))
print('WB inhibitory cells:', len(rtm_model.area_names))
print('Stimulated area:', rtm_model.area_names[v4_index])
print('Local synapses: RTM → WB and WB → RTM in each area')
print('No inter-area synapses are created.')


## Step 6: Run the Local Circuits

This runs both cell types and their within-area synapses. Areas remain disconnected from one another.


In [ ]:
rtm_model.run()
print("RTM simulation complete.")


## Step 7: Inspect Voltage Results

Both voltage arrays have rows for time points and columns for brain areas. Compare RTM and WB responses in V4.


In [ ]:
time_ms = rtm_model.time_ms
voltage_mv = rtm_model.voltage_mv
wb_voltage_mv = rtm_model.wb_voltage_mv
spike_times_by_area = rtm_model.spike_times_by_area
wb_spike_times_by_area = rtm_model.wb_spike_times_by_area

print('RTM voltage shape (time points, areas):', voltage_mv.shape)
print('WB voltage shape (time points, areas):', wb_voltage_mv.shape)
print('RTM voltage range:', np.nanmin(voltage_mv), 'to', np.nanmax(voltage_mv), 'mV')
print('WB voltage range:', np.nanmin(wb_voltage_mv), 'to', np.nanmax(wb_voltage_mv), 'mV')


## Step 8: Plot V4 Input and Both Cell Voltages

The upper plot shows the external pulse to V4. The lower plot overlays the RTM excitatory-cell voltage and WB inhibitory-cell voltage, so you can see how local inhibition follows excitation.


In [ ]:
input_trace = rtm_model.input_current_uA_cm2[:, v4_index]
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(time_ms, input_trace)
axes[0].set_ylabel('Input current (µA/cm²)')
axes[0].set_title('Current delivered to V4 RTM cell')
axes[1].plot(time_ms, voltage_mv[:, v4_index], label='RTM excitatory')
axes[1].plot(time_ms, wb_voltage_mv[:, v4_index], label='WB inhibitory', alpha=0.8)
axes[1].set_ylabel('Membrane voltage (mV)')
axes[1].set_xlabel('Time (ms)')
axes[1].set_title('Local RTM–WB response in V4')
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## Step 9: Compare RTM Outputs Across Areas

Only V4 receives the external pulse. The other areas have the same baseline input and the same local RTM–WB circuit, so their traces should overlap. Activity does not propagate between areas because there are no inter-area connections.


In [ ]:
n_show = min(5, n_areas)
plt.figure(figsize=(9, 4))
indices_to_show = [area_labels.index("V4")] + [
    idx for idx, label in enumerate(area_labels) if label != "V4"
][:n_show - 1]
for idx in indices_to_show:
    plt.plot(time_ms, voltage_mv[:, idx], label=area_labels[idx], alpha=0.8)
plt.xlabel("Time (ms)")
plt.ylabel("Membrane voltage (mV)")
plt.title("RTM signals across areas")
plt.legend()
plt.grid(alpha=0.25)
plt.show()
print("Maximum voltage difference among the displayed areas:",
      np.max(np.ptp(voltage_mv[:, indices_to_show], axis=1)), "mV")


## Step 10: View RTM and WB Spike Times

The model detects spikes when voltage crosses 0 mV. A reset to −65 mV marks each spike for synaptic event delivery. This threshold-and-reset mechanism is an implementation choice and can affect the voltage waveform compared with the original MATLAB integration.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for idx, spike_times in enumerate(spike_times_by_area):
    axes[0].scatter(spike_times, np.full(len(spike_times), idx), s=12, color='tab:blue')
for idx, spike_times in enumerate(wb_spike_times_by_area):
    axes[1].scatter(spike_times, np.full(len(spike_times), idx), s=12, color='tab:orange')
axes[0].set_ylabel('Area index')
axes[0].set_title('RTM excitatory spikes')
axes[1].set_ylabel('Area index')
axes[1].set_xlabel('Time (ms)')
axes[1].set_title('WB inhibitory spikes')
for ax in axes:
    ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()
print('RTM spike counts (first areas):', [len(x) for x in spike_times_by_area[:10]])
print('WB spike counts (first areas):', [len(x) for x in wb_spike_times_by_area[:10]])


## Scope of This Model

- Each area has one RTM excitatory cell and one WB inhibitory interneuron.
- RTM-to-WB excitation and WB-to-RTM inhibition are local to each area.
- There are no connections between areas, and the anatomical FLN matrix is not used yet.
- This is a two-cell local-circuit prototype, not a large neuronal population.
- The MATLAB files do not specify the current units; the notebook provisionally assumes µA/cm².
- Brian2 uses fixed-step integration, so traces can differ from MATLAB's ode15s results.
